# 🕊️ Make your Christian-AI training data — no terminal needed

This notebook does **Phase 1** for you. It turns the CAB-FF benchmark into
training data and then downloads two files to your computer.

## You only do 3 things:
1. ✏️ In **STEP 1** below, paste your API key into the box.
2. 🔽 Pick `openai` or `anthropic` for the key type.
3. ▶️ Click the menu at the top: **Runtime → Run all**. Then wait ~30 min.

When it finishes, **two files download automatically** (`train.jsonl` and
`eval.jsonl`) — those are what you upload to AutoTrain in Phase 2.

> ⚠️ **This costs money on your API key** — roughly **$3–5** with an OpenAI
> key (cheapest) or **$15–30** with a Claude key. That's because an AI is
> writing thousands of answers for you. There's no way around this part;
> the answers have to be written by an AI, and that AI charges per use.

> 💡 You do **not** need a fancy computer. Colab runs in the cloud. You can
> close your laptop lid as long as the tab stays open... actually, keep the
> tab open and the laptop awake until it finishes.

In [ ]:
#@title ✏️ STEP 1 — paste your API key, then pick the type

MY_KEY = "PASTE-YOUR-KEY-HERE"  #@param {type:"string"}

# Where did your key come from?
#   openai    = platform.openai.com   (key starts with sk-...,  CHEAPEST ~$3-5)
#   anthropic = console.anthropic.com (key starts with sk-ant-..., ~$15-30)
KEY_TYPE = "openai"  #@param ["openai", "anthropic"]

In [ ]:
#@title ▶️ STEP 2 — run everything (Runtime → Run all, or press play here)

import os, sys, subprocess

assert MY_KEY and "PASTE" not in MY_KEY, \
    "⛔ Go back to STEP 1 and paste your real API key first!"

print("⏳ (1/4) Downloading the project...")
if not os.path.isdir("SoliDeoGloria"):
    subprocess.run(["git", "clone", "--quiet",
                    "https://github.com/moonshineaitech/SoliDeoGloria"], check=True)
os.chdir("/content/SoliDeoGloria/trainer")

print("⏳ (2/4) Installing the data tools (~2 min, lots of output is normal)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[teachers]", "-q"], check=True)

if KEY_TYPE == "openai":
    os.environ["OPENAI_API_KEY"] = MY_KEY
    TEACHER = "gpt-4o-mini"
else:
    os.environ["ANTHROPIC_API_KEY"] = MY_KEY
    TEACHER = "claude-haiku-4-5-20251001"

print(f"⏳ (3/4) Writing training data with teacher = {TEACHER} (~30 min, be patient)...")
subprocess.run([sys.executable, "-m", "trainer.data.pipeline.cli", "build",
                "--dataset", "../data/CAB_FF_v3_dataset.json",
                "--teacher", TEACHER,
                "--max-synth", "800", "--max-prefs", "400",
                "--out", "data/built"], check=True)

print("⏳ (4/4) Sending the files to your Downloads folder...")
from google.colab import files
for f in ["train.jsonl", "eval.jsonl", "pref_train.jsonl", "pref_eval.jsonl"]:
    path = f"data/built/{f}"
    if os.path.exists(path):
        files.download(path)

print("\n✅ DONE! Check your browser's Downloads folder.")
print("   Upload train.jsonl and eval.jsonl to AutoTrain (Phase 2).")

## 🆘 If something goes wrong

| What you see | What to do |
|---|---|
| Stops on STEP 1 with "paste your real key" | You didn't replace `PASTE-YOUR-KEY-HERE`. Edit STEP 1 and run again. |
| An error mentioning `git clone` / `Repository not found` | The repo may be private — tell Claude and it'll give you a clone link with access. |
| An error mentioning `rate limit` or `insufficient quota` | Add a little more credit to your API account, then **Runtime → Run all** again. |
| The tab disconnected | Reopen it and **Runtime → Run all** again — it picks back up. |
| No download popped up | Click the 📁 folder icon on the left → `SoliDeoGloria/trainer/data/built/` → download `train.jsonl` and `eval.jsonl` by hand. |

*Soli Deo Gloria.*